In [1]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.3/251.3 kB 2.4 MB/s eta 0:00:0000:01


In [2]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 12.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
from multiprocessing import Pool
from joblib import Parallel, delayed
import itertools

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('../../parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [17]:
fn = '../../Active_SAM_joined/SAM_MO_soupx_plus5_cleaned_NN_04142026.h5ad'

In [18]:
sam=SAM()
sam.load_data(fn)
gene_dict = {}
for i in range(len(sam.adata.var_names)):
    gene_dict[sam.adata.var_names[i]] = i

In [19]:
sam.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_counts', 'n_genes',
       'key', 'subclass_id_label_mapping', 'subclass_id_label_lc',
       'leiden_clusters', 'subclass_id_label_mapping_nounlabeled',
       'neurotransmitter', 'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled',
       'leiden_clusters_formarkers', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'NN', 'ss_subclass',
       'ss_subclass_nounlabeled', 'ss_subclass_nounlabeled_03102026',
       'ss_subclass_nounlabeled_03102026_crossed', 'neurotransmitter_v2'],
      dtype='object')

In [20]:
level = 'ss_subclass_nounlabeled_03102026'

In [21]:
for item in sam.adata.obs[level].unique():
    if item not in parent_dict.keys():
        parent_dict[item] = 'hypo'
    if item[:2] == 'mo':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mg':
        parent_dict[item] = 'hypo'
    if item[:2] == 'xt':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ac':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cj':
        parent_dict[item] = 'hypo'
    if item[:2] == 'dr':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cc':
        parent_dict[item] = 'hypo'
    if item[:2] == 'rv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[-2:] == 'NN':
        parent_dict[item] = 'Non neuron'

In [22]:
parent_dict['not hypo'] = 'not hypo'
parent_dict['hypo'] = 'hypo'
parent_dict['Unknown'] = 'hypo'
parent_dict['Unknwon_2'] = 'hypo'
parent_dict['Non neuron'] = 'not hypo'

In [23]:
ordered_genes = sam.identify_marker_genes_ratio(level)

In [24]:
ordered_genes.keys()

dict_keys(['007 L2/3 IT CTX Glut', '009 L2/3 IT PIR-ENTl Glut', '010 IT AON-TT-DP Glut', '014 LA-BLA-BMA-PA Glut', '016 CA1-ProS Glut', '017 CA3 Glut', '022 L5 ET CTX Glut', '037 DG Glut', '041 OB-in Frmd7 Gaba', '047 Sncg Gaba', '049 Lamp5 Gaba', '054 STR Prox1 Lhx6 Gaba', '055 STR Lhx8 Gaba', '056 Sst Chodl Gaba', '057 NDB-SI-MA-STRv Lhx8 Gaba', '058 PAL-STR Gaba-Chol', '059 GPe-SI Sox6 Cyp26b1 Gaba', '061 STR D1 Gaba', '062 STR D2 Gaba', '063 STR D1 Sema5a Gaba', '064 STR-PAL Chst9 Gaba', '066 NDB-SI-ant Prdm12 Gaba', '067 LSX Sall3 Pax6 Gaba', '068 LSX Otx2 Gaba', '069 LSX Nkx2-1 Gaba', '070 LSX Prdm12 Slit2 Gaba', '073 MEA-BST Sox6 Gaba', '074 MEA-BST Lhx6 Sp9 Gaba', '075 MEA-BST Lhx6 Nr2e1 Gaba', '076 MEA-BST Lhx6 Nfib Gaba', '077 CEA-BST Gal Avp Gaba', '078 SI-MA-ACB Ebf1 Bnc2 Gaba', '079 CEA-BST Six3 Cyp26b1 Gaba', '080 CEA-AAA-BST Six3 Sp9 Gaba', '081 ACB-BST-FS D1 Gaba', '082 CEA-BST Ebf1 Pdyn Gaba', '083 CEA-BST Rai14 Pdyn Crh Gaba', '084 BST-SI-AAA Six3 Slc22a3 Gaba', '085 

In [25]:
hypo_ct = [i for i in sam.adata.obs[level].unique() if parent_dict[parent_dict[i]] == 'hypo']

In [26]:
np.sort(hypo_ct)

array(['066 NDB-SI-ant Prdm12 Gaba', '073 MEA-BST Sox6 Gaba',
       '074 MEA-BST Lhx6 Sp9 Gaba', '075 MEA-BST Lhx6 Nr2e1 Gaba',
       '076 MEA-BST Lhx6 Nfib Gaba', '077 CEA-BST Gal Avp Gaba',
       '078 SI-MA-ACB Ebf1 Bnc2 Gaba', '079 CEA-BST Six3 Cyp26b1 Gaba',
       '080 CEA-AAA-BST Six3 Sp9 Gaba', '081 ACB-BST-FS D1 Gaba',
       '082 CEA-BST Ebf1 Pdyn Gaba', '083 CEA-BST Rai14 Pdyn Crh Gaba',
       '084 BST-SI-AAA Six3 Slc22a3 Gaba', '085 SI-MPO-LPO Lhx8 Gaba',
       '086 MPO-ADP Lhx8 Gaba', '087 MPN-MPO-LPO Lhx6 Zfhx3 Gaba',
       '088 BST Tac2 Gaba', '089 PVR Six3 Sox3 Gaba',
       '090 BST-MPN Six3 Nrgn Gaba', '091 ARH-PVi Six6 Dopa-Gaba',
       '092 TMv-PMv Tbx3 Hist-Gaba', '093 RT-ZI Gnb3 Gaba',
       '094 SCH Six6 Cdc14a Gaba', '097 PVHd-SBPV Six3 Prox1 Gaba',
       '098 AHN-SBPV-PVHd Pdrm12 Gaba', '099 SBPV-PVa Six6 Satb2 Gaba',
       '100 AHN Onecut3 Gaba', '101 ZI Pax6 Gaba',
       '102 DMH-LHA Gsx1 Gaba', '103 PVHd-DMH Lhx6 Gaba',
       '104 TU-ARH Otp Six6 

In [27]:
#Check claude 
def markers_celltype_neighbor(cto, neigh_cto, test_top, diff, qdiffth, qdiffth2,
                              num_ct_better, A, obs_level, ordered_genes, gene_dict):
    mask = obs_level == cto
    ct_X = A[mask, :]
    bkgd_all_X = A[np.isin(obs_level, ref) & ~mask,:]                           # all cells NOT in cto
    neigh_X = {ctt: A[obs_level == ctt, :] for ctt in neigh_cto}  # precompute once

    markers = []
    for gene in ordered_genes[cto][:test_top]:
        gene_index = gene_dict[gene]
        ct_exp_g = ct_X[:, gene_index]
        fct_exp_g = np.sum(ct_exp_g > 0) / len(ct_exp_g)
        ct_mean = np.average(ct_exp_g)

        num = 0
        sig = None  # significance vs full background, computed at most once per gene
        for ctt in neigh_cto:
            bkgd_exp_g = neigh_X[ctt][:, gene_index]
            fbkgd_exp_g = np.sum(bkgd_exp_g > 0) / len(bkgd_exp_g)

            # effect size is still measured relative to the close neighbor
            crit = ((fct_exp_g > diff and (fct_exp_g - fbkgd_exp_g) / fct_exp_g > qdiffth)
                    or (ct_mean - np.average(bkgd_exp_g) > qdiffth2))
            if crit:
                if sig is None:
                    st, pval = stats.mannwhitneyu(x=ct_exp_g,
                                                  y=bkgd_all_X[:, gene_index],
                                                  alternative='greater')
                    sig = pval < .05 / test_top
                if sig:
                    num += 1

        if num >= num_ct_better:
            markers.append(gene)
    return cto, markers

In [28]:
#Check claude
test_top = 7000
diff = .01
qdiffth = .8
qdiffth2 = 1
kn = 10
num_ct_better = 1
query = hypo_ct
ref = list(hypo_ct)

A = sam.adata.X.A
obs_level = sam.adata.obs[level].values
pca = sam.adata.obsm['X_pca']

# --- precompute nearest-neighbor cell types once (centroids in PCA space) ---
t0 = time.time()
centroids = np.vstack([pca[obs_level == ct].mean(axis=0) for ct in ref])
neigh = {}
for q in query:
    qc = pca[obs_level == q].mean(axis=0)
    d = np.linalg.norm(centroids - qc, axis=1)
    di = np.argsort(d)[:kn + 1][1:]          # drop self (nearest, distance 0)
    neigh[q] = [ref[i] for i in di]
print('created neighbors in: ' + str(time.time() - t0) + ' seconds')

# --- parallel marker calling ---
t1 = time.time()
results = Parallel(n_jobs=8, verbose=10)(
    delayed(markers_celltype_neighbor)(cto,
                                       neigh[cto],
                                       test_top=test_top,
                                       diff=diff,
                                       qdiffth=qdiffth,
                                       qdiffth2=qdiffth2,
                                       num_ct_better=num_ct_better,
                                       A=A,
                                       obs_level=obs_level,
                                       ordered_genes=ordered_genes,
                                       gene_dict=gene_dict)
    for cto in query
)
marker_dict = dict(results)
print('called markers in: ' + str(time.time() - t1) + ' seconds')
for cto in query:
    print(cto, len(marker_dict[cto]))

created neighbors in: 0.12327957153320312 seconds


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   28.7s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:   47.6s
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:  1.2min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  1.6min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:  2.0min
[Parallel(n_jobs=8)]: Done  45 tasks      | elapsed:  2.5min
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  3.3min
[Parallel(n_jobs=8)]: Done  69 tasks      | elapsed:  4.0min
[Parallel(n_jobs=8)]: Done  82 tasks      | elapsed:  4.6min
[Parallel(n_jobs=8)]: Done  94 out of  99 | elapsed:  5.6min remaining:   18.0s
[Parallel(n_jobs=8)]: Done  99 out of  99 | elapsed:  6.4min finished


called markers in: 383.89653515815735 seconds
mo_5 68
mo_0 59
108 ARH-PVp Tbx3 Gaba 175
mo_23 117
128 VMH Fezf1 Glut 160
140 PMd-LHA Foxb1 Glut 217
097 PVHd-SBPV Six3 Prox1 Gaba 132
107 DMH Hmx2 Gaba 96
101 ZI Pax6 Gaba 125
086 MPO-ADP Lhx8 Gaba 180
mo_6 70
mg_077_106 103
135 STN-PSTN Pitx2 Glut 123
133 PVH-SO-PVa Otp Glut 282
092 TMv-PMv Tbx3 Hist-Gaba 213
103 PVHd-DMH Lhx6 Gaba 114
132 AHN-RCH-LHA Otp Fezf1 Glut 125
099 SBPV-PVa Six6 Satb2 Gaba 171
127 DMH-LHA Vgll2 Glut 243
mo_9 57
mo_14 106
106 PVpo-VMPO-MPN Hmx2 Gaba 146
mo_4 95
144 MM Foxb1 Glut 469
129 VMH Nr5a1 Glut 189
116 AVPV-MEPO-SFO Tbr1 Glut 215
098 AHN-SBPV-PVHd Pdrm12 Gaba 145
mo_2 94
105 TMd-DMH Foxd2 Gaba 192
143 MM-ant Foxb1 Glut 378
mo_10 104
100 AHN Onecut3 Gaba 77
136 PMv-TMv Pitx2 Glut 139
093 RT-ZI Gnb3 Gaba 190
123 DMH Nkx2-4 Glut 392
mo_11 45
117 LHA Barhl2 Glut 78
134 PH-ant-LHA Otp Bsx Glut 125
119 SI-MA-LPO-LHA Skor1 Glut 107
118 ADP-MPO Trp73 Glut 113
076 MEA-BST Lhx6 Nfib Gaba 316
124 MPN-MPO-PVpo Hmx2 Gl

In [80]:
folder_name = 'MO_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07272026'
if not os.path.isdir('../../Important_genes/' + folder_name):
    os.mkdir('../../Important_genes/' + folder_name)
df = pd.DataFrame(data = [qdiffth,qdiffth2, diff, kn, num_ct_better], index = ['qdiffth','qdiffth2', 'diff', 'kn', 'num_ct_better']) 
df.to_csv('../../Important_genes/' + folder_name + '/metadata.csv')
with open('../../Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
    pickle.dump(marker_dict, f)